# Projekt 01 (basic) — Ein STRIPS-Vorwaertsplaner

**Modul 07 — Theorie der KI 2**

Du baust einen **klassischen Planer**: STRIPS-Zustaende und -Aktionen, Vorwaerts-
suche (Progression) mit BFS *und* mit A\* unter der **h_add-Heuristik** aus der
Delete-Relaxation (Skript Teil 1). Testfall ist die beruehmte **Sussman-Anomalie**
der Blocksworld — ein kleines Problem, an dem naive Teilziel-Planer scheitern,
Zustandssuche aber nicht.

Ziel: verstehen, wie sich Planung als Suche (Modul 06) formulieren laesst und wie
eine **automatisch aus der Aktionsbeschreibung gewonnene Heuristik** die Suche lenkt.

## Setup
Nur Standardbibliothek. Kernel des Repo-`.venv` waehlen (siehe `SETUP.md`) und die
Zellen von oben nach unten ausfuehren. Loese dann **Aufgabe 1–3** (die `TODO`-Zellen).

In [1]:
import heapq, itertools
from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, Tuple, Any
print("Bibliotheken geladen (nur Standardbibliothek).")

Bibliotheken geladen (nur Standardbibliothek).


## Teil A — STRIPS & Blocksworld (vorgegeben)
Die Repraesentation: Fluenten als Tupel, Zustaende als `frozenset`, Aktionen als
`Action(pre, add, dele)`. Progression ist reine Mengenarithmetik
$(s\setminus\mathrm{DEL})\cup\mathrm{ADD}$. Lies und fuehre die Zellen aus.

In [2]:
# ---- STRIPS-Repraesentation ------------------------------------------------
# Ein Fluent ist ein Tupel, z.B. ("on","C","A") oder ("handempty",).
# Ein Zustand ist ein frozenset von Fluenten (Closed-World: was fehlt, ist falsch).

@dataclass(frozen=True)
class Action:
    name: str
    pre: FrozenSet
    add: FrozenSet
    dele: FrozenSet          # "del" ist Python-Keyword -> dele
    def __repr__(self): return self.name

def applicable(action, state):
    """Anwendbar, wenn alle Vorbedingungen im Zustand gelten."""
    return action.pre <= state

def result(state, action):
    """Progression:  (s ohne DEL) vereinigt ADD."""
    return frozenset((state - action.dele) | action.add)

# ---- Blocksworld: Aktionsschemata fuer konkrete Bloecke instanziieren -------
def blocksworld_actions(blocks):
    acts = []
    for x in blocks:
        acts.append(Action(f"PickUp({x})",
            pre=frozenset({("clear", x), ("ontable", x), ("handempty",)}),
            add=frozenset({("holding", x)}),
            dele=frozenset({("clear", x), ("ontable", x), ("handempty",)})))
        acts.append(Action(f"PutDown({x})",
            pre=frozenset({("holding", x)}),
            add=frozenset({("ontable", x), ("clear", x), ("handempty",)}),
            dele=frozenset({("holding", x)})))
        for y in blocks:
            if x == y:
                continue
            acts.append(Action(f"Stack({x},{y})",
                pre=frozenset({("holding", x), ("clear", y)}),
                add=frozenset({("on", x, y), ("clear", x), ("handempty",)}),
                dele=frozenset({("holding", x), ("clear", y)})))
            acts.append(Action(f"Unstack({x},{y})",
                pre=frozenset({("on", x, y), ("clear", x), ("handempty",)}),
                add=frozenset({("holding", x), ("clear", y)}),
                dele=frozenset({("on", x, y), ("clear", x), ("handempty",)})))
    return acts

@dataclass(frozen=True)
class PlanProblem:
    initial: FrozenSet
    goal: FrozenSet
    actions: Tuple
    def is_goal(self, state): return self.goal <= state
    def successors(self, state):
        return [(a, result(state, a)) for a in self.actions if applicable(a, state)]

# ---- Die Sussman-Anomalie (der beruehmte Blocksworld-Testfall) -------------
BLOCKS = ["A", "B", "C"]
s0 = frozenset({("on", "C", "A"), ("ontable", "A"), ("ontable", "B"),
                ("clear", "C"), ("clear", "B"), ("handempty",)})
goal = frozenset({("on", "A", "B"), ("on", "B", "C")})
problem = PlanProblem(s0, goal, tuple(blocksworld_actions(BLOCKS)))

def show_state(s):
    parts = []
    for f in sorted(s):
        parts.append(f[0] + ("(" + ",".join(f[1:]) + ")" if len(f) > 1 else ""))
    return "  ".join(parts)

print("Start :", show_state(s0))
print("Ziel  :", show_state(goal))
print("Grundaktionen:", len(problem.actions))

Start : clear(B)  clear(C)  handempty  on(C,A)  ontable(A)  ontable(B)
Ziel  : on(A,B)  on(B,C)
Grundaktionen: 18


## Teil B — Suchinfrastruktur & BFS (vorgegeben)
Derselbe `Node` wie in Modul 06 und die generische Bestensuche. BFS ist als
Vorbild vollstaendig gegeben.

In [3]:
# ---- Suchknoten (wie in Modul 06) ------------------------------------------
@dataclass(frozen=True)
class Node:
    state: Any
    parent: Any = None
    action: Any = None
    path_cost: float = 0.0     # = Anzahl Aktionen bis hier
    def plan(self):
        acts, n = [], self
        while n.parent is not None:
            acts.append(n.action); n = n.parent
        return list(reversed(acts))

print("Suchknoten bereit.")

Suchknoten bereit.


In [4]:
# ---- Vorwaertssuche mit BFS (vollstaendig vorgegeben) ----------------------
def bfs_plan(problem):
    node = Node(problem.initial)
    if problem.is_goal(node.state):
        return node, 0
    frontier = deque([node]); reached = {node.state}; expanded = 0
    while frontier:
        node = frontier.popleft(); expanded += 1
        for a, s2 in problem.successors(node.state):
            if s2 in reached:
                continue
            child = Node(s2, node, a, node.path_cost + 1)
            if problem.is_goal(s2):
                return child, expanded
            reached.add(s2); frontier.append(child)
    return None, expanded

node, exp = bfs_plan(problem)
print("BFS-Plan (", len(node.plan()), "Aktionen ), expandierte Zustaende:", exp)
for i, a in enumerate(node.plan(), 1):
    print(f"  {i}. {a}")

BFS-Plan ( 6 Aktionen ), expandierte Zustaende: 18
  1. Unstack(C,A)
  2. PutDown(C)
  3. PickUp(B)
  4. Stack(B,C)
  5. PickUp(A)
  6. Stack(A,B)


In [5]:
# ---- Generische Bestensuche (vorgegeben) -----------------------------------
def best_first_plan(problem, f):
    node = Node(problem.initial)
    counter = itertools.count()
    frontier = [(f(node.state, 0), next(counter), node)]
    reached = {problem.initial: 0}
    expanded = 0
    while frontier:
        _, _, node = heapq.heappop(frontier)
        if problem.is_goal(node.state):
            return node, expanded
        expanded += 1
        for a, s2 in problem.successors(node.state):
            g2 = node.path_cost + 1
            if s2 not in reached or g2 < reached[s2]:
                reached[s2] = g2
                child = Node(s2, node, a, g2)
                heapq.heappush(frontier, (f(s2, g2), next(counter), child))
    return None, expanded

print("best_first_plan bereit — A* ist die richtige Wahl von f.")

best_first_plan bereit — A* ist die richtige Wahl von f.


### Aufgabe 1 — die h_add-Heuristik (Delete-Relaxation)
Implementiere `h_add`: Ignoriere alle Delete-Listen und schaetze per Fixpunkt-
Iteration die Kosten jedes Fluents; summiere ueber die Zielfluenten. Siehe die
Formel im Zellenkommentar und Skript Teil 1.4.

In [6]:
# ---- Delete-Relaxation: die h_add-Heuristik --------------------------------
def h_add(state, problem):
    """Geschaetzte Kosten, das Ziel zu erreichen, unter Ignorieren aller
    Delete-Listen. Delta(p)=0 fuer p in state, sonst
    min ueber Aktionen a mit p in ADD(a) von (1 + sum_{q in PRE(a)} Delta(q)).
    Fixpunkt-Iteration. h_add = Summe der Delta ueber die Zielfluenten."""
    INF = float("inf")
    delta = {f: 0 for f in state}
    changed = True
    while changed:
        changed = False
        for a in problem.actions:
            if any(delta.get(q, INF) == INF for q in a.pre):
                continue
            cost_a = 1 + sum(delta[q] for q in a.pre)
            for p in a.add:
                if cost_a < delta.get(p, INF):
                    delta[p] = cost_a; changed = True
    total = sum(delta.get(g, INF) for g in problem.goal)
    return total

print("h_add(Start) =", h_add(problem.initial, problem))

h_add(Start) = 5


### Aufgabe 2 — A* mit h_add
Verdrahte `best_first_plan` mit $f(s,g)=g+h_{\text{add}}(s)$ zu A\*.

In [7]:
# ---- A* mit h_add ----------------------------------------------------------
def astar_plan(problem):
    return best_first_plan(problem, f=lambda s, g: g + h_add(s, problem))

node, exp = astar_plan(problem)
print("A*-Plan (", len(node.plan()), "Aktionen ), expandierte Zustaende:", exp)
for i, a in enumerate(node.plan(), 1):
    print(f"  {i}. {a}")

A*-Plan ( 6 Aktionen ), expandierte Zustaende: 11
  1. Unstack(C,A)
  2. PutDown(C)
  3. PickUp(B)
  4. Stack(B,C)
  5. PickUp(A)
  6. Stack(A,B)


### Aufgabe 3 — Vergleich
Vergleiche BFS und A\*(h_add): Planlaenge und Zahl expandierter Zustaende. Beide
sollen einen optimalen **6-Schritte-Plan** finden; A\* expandiert weniger.

In [8]:
# ---- Vergleich BFS vs. A*(h_add) -------------------------------------------
b_node, b_exp = bfs_plan(problem)
a_node, a_exp = astar_plan(problem)
print(f"BFS       : Planlaenge {len(b_node.plan())}, {b_exp} Zustaende expandiert")
print(f"A*(h_add) : Planlaenge {len(a_node.plan())}, {a_exp} Zustaende expandiert")
assert len(b_node.plan()) == len(a_node.plan()) == 6, "Sussman-Optimum ist 6 Aktionen"
print("\nBeide finden einen optimalen 6-Schritte-Plan; die Heuristik spart Expansionen.")
print("(Sussman-Anomalie: das Ziel zerfaellt NICHT in unabhaengige Teilziele —")
print(" ein naiver Teilziel-Planer scheitert, die Zustandssuche nicht.)")

BFS       : Planlaenge 6, 18 Zustaende expandiert
A*(h_add) : Planlaenge 6, 11 Zustaende expandiert

Beide finden einen optimalen 6-Schritte-Plan; die Heuristik spart Expansionen.
(Sussman-Anomalie: das Ziel zerfaellt NICHT in unabhaengige Teilziele —
 ein naiver Teilziel-Planer scheitert, die Zustandssuche nicht.)


## Teil C — Verifikation
Zum Schluss pruefen wir, dass der gefundene Plan Schritt fuer Schritt anwendbar
ist und das Ziel erreicht.

In [9]:
# ---- Selbstcheck: ist der A*-Plan wirklich ausfuehrbar und zielfuehrend? ----
s = problem.initial
for a in a_node.plan():
    assert applicable(a, s), f"Aktion {a} nicht anwendbar!"
    s = result(s, a)
assert problem.is_goal(s), "Endzustand erfuellt das Ziel nicht!"
print("Verifiziert: der Plan ist Schritt fuer Schritt ausfuehrbar und erreicht das Ziel.")
print("Endzustand:", show_state(s))

Verifiziert: der Plan ist Schritt fuer Schritt ausfuehrbar und erreicht das Ziel.
Endzustand: clear(A)  handempty  on(A,B)  on(B,C)  ontable(C)


## Reflexion (kurz, schriftlich)
1. Warum ist die Sussman-Anomalie fuer einen Planer schwierig, der das Ziel in
   unabhaengige Teilziele zerlegt und nacheinander loest?
2. `h_add` ist **nicht** zulaessig (kann ueberschaetzen). Wieso stoert das die
   Optimalitaet deines A\* hier trotzdem nicht bzw. wann wuerde es stoeren?
   (Tipp: vergleiche mit $h_{\max}$.)
3. Wie wuerde sich die Suche aendern, wenn du **Regression** (Rueckwaerts) statt
   Progression benutzt? (Skript Teil 1.3.)
4. Woher „weiss" `h_add` etwas ueber die Domaene, obwohl es domaenen-*unabhaengig*
   ist? (Stichwort: es liest die ADD/PRE-Struktur der Aktionen.)

Musterantworten am Ende der Loesung im Ordner `solution/`.